# VideoDB Understanding: Two-Pass Action Understanding

Analyze a video in 5-second intervals:

- **Pass 1 — evidence:** combine 5 frames into one image and describe what happens.
- **Pass 2 — actions:** combine 10 frames into one image, then use neighboring evidence and previous action state to build an action timeline.

You can read the output of either pass.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/two-pass-action-understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install dependencies

In [ ]:
%pip install -q "videodb>=0.5.0" python-dotenv pandas

## 2. Connect to VideoDB

Set `VIDEO_DB_BASE_URL` in your environment or `.env` to the Runtime endpoint with two-pass support enabled.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")

conn = connect(api_key=api_key, base_url=os.environ["VIDEO_DB_BASE_URL"])
collection = conn.get_collection()

## 3. Choose a video

Replace the sample URL with your own video, or reuse an uploaded video.

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"
video = collection.upload(url=VIDEO_URL)

# To reuse an uploaded video, replace the upload above with:
# video = collection.get_video("YOUR_VIDEO_ID")

video.play()

## 4. Run the two passes

`inputs` connects the passes. `context.inputs` selects neighboring evidence, while `context.state` carries action state between intervals.

Change `seconds`, `every`, or `frames_per_sheet` below to try different settings.

In [ ]:
MODEL = "google/gemini-3.8-flash"  # or "google/gemini-3.7-flash"

understanding = video.understand(
    segmentation={"type": "time", "seconds": 5},
    analyzers=[
        {
            "type": "vlm",
            "name": "evidence",
            "sampling": {"strategy": "interval", "every": 1},
            "preprocessing": [
                {"type": "contact_sheet", "frames_per_sheet": 5, "columns": 3}
            ],
            "config": {
                "model": MODEL,
                "prompt": "Describe the visible actions and how people interact with objects.",
            },
        },
        {
            "type": "vlm",
            "name": "actions",
            "inputs": ["evidence"],
            "sampling": {"strategy": "interval", "every": 0.5},
            "preprocessing": [
                {
                    "type": "contact_sheet",
                    "frames_per_sheet": 10,
                    "columns": 5,
                    "labels": ["frame_id"],
                }
            ],
            "config": {
                "model": MODEL,
                "prompt": "Identify continuous actions and task stages. Keep an action open if it continues.",
                "context": {
                    "inputs": {"evidence": {"previous": 1, "current": True, "next": 1}},
                    "state": {
                        "policy": "action_timeline_v1",
                        "recent_segments": 3,
                        "recent_stages": 2,
                    },
                },
            },
        },
    ],
)

print("Understanding ID:", understanding.id)

In [ ]:
understanding.wait_until_complete(timeout=7200, poll_interval=15)
print("Status:", understanding.status)

## 5. View the results

Evidence contains descriptions for each interval. Actions can span several intervals.

In [ ]:
import pandas as pd

evidence_output = understanding.get_analyzer("evidence").get_output()
pd.json_normalize(evidence_output["scenes"]).head()

In [ ]:
actions_output = understanding.get_analyzer("actions").get_output()
pd.json_normalize(actions_output["scenes"]).head()